# 🎨 ComfyUI Colab — Qwen Multi-Angle

Generate multi-angle product shots from a single image using ComfyUI + Qwen Image Edit.

**Run cells top to bottom.** Code is hidden by default — expand to edit.

---
| Step | What it does |
|------|-------------|
| 1 · Permissions | Mounts Google Drive + grants HF token access |
| 2 · Install | ComfyUI + speed stack (xformers, triton, sageattention) |
| 3 · Custom Nodes | Install extra nodes from GitHub |
| 4 · Models | Download Qwen multi-angle model files |
| 5 · Launch | Starts ComfyUI + watches input folder for new images |

---
### ▶️ How to run

1. Upload product images to your Drive input folder — `<your root folder>/input/`
2. Place `workflow_api.json` in `<your root folder>/`
3. Run **Runtime → Run all**
4. Approve the **one permission popup** (Google Drive + HF Token) at the top
5. Wait for all images to process — outputs appear in `<your root folder>/output/`
6. When done, go to **Runtime → Disconnect and delete runtime** to stop billing

---


In [2]:
# @title 🔐 1 · Permissions — Google Drive + HF Token { display-mode: "form" }
# @markdown Mounts Google Drive and grants access to your HF_TOKEN secret.
# @markdown **This is the only cell that shows a permission popup.**
# @markdown Run this before anything else.

from google.colab import drive, userdata
import os

# ── HF Token (triggers grant popup if not yet approved) ──────────
print("🔑 Requesting HF_TOKEN access...")
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("✅ HF_TOKEN loaded from Colab Secrets")
except userdata.SecretNotFoundError:
    print("⚠️  HF_TOKEN not found — public models will still download fine")
    print("   (Add it via the 🔑 Secrets panel if you need private model access)")
except userdata.NotebookAccessError:
    print("⚠️  HF_TOKEN access denied — continuing without it")

# ── Mount Google Drive ────────────────────────────────────────────
print("\n📂 Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=False)

if os.path.exists("/content/drive/Shareddrives"):
    print("✅ Shared Drives detected")
elif os.path.exists("/content/drive/MyDrive"):
    print("✅ MyDrive detected")
else:
    print("⚠️  Drive mounted but no folders found — check permissions")

print("\n✅ Permissions granted — all other cells run silently.")


🔑 Requesting HF_TOKEN access...
✅ HF_TOKEN loaded from Colab Secrets

📂 Connecting to Google Drive...
Mounted at /content/drive
✅ Shared Drives detected

✅ Permissions granted — all other cells run silently.


In [3]:
# @title ⚙️ 2 · Install ComfyUI & Dependencies { display-mode: "form" }
# @markdown ## Install ComfyUI & Dependencies
# @markdown
# @markdown Clones ComfyUI and installs the full stack including the **speed trio**:
# @markdown - **xformers** — faster attention, lower VRAM
# @markdown - **triton** — GPU kernel acceleration
# @markdown - **sageattention** — additional attention optimization
# @markdown
# @markdown Also sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to reduce OOM crashes.

import os
from pathlib import Path

WORKSPACE = "/content/ComfyUI"

# ── 0. Install uv ─────────────────────────────────────────────────
print("📦 Installing uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
print("✅ uv ready\n")

# ── 1. Clone / update ComfyUI ─────────────────────────────────────
if not os.path.exists(WORKSPACE):
    print("📥 Cloning ComfyUI...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
    print("✅ Cloned\n")
else:
    print("✅ ComfyUI exists, pulling updates...")
    !cd {WORKSPACE} && git pull -q
    print("✅ Updated\n")

%cd {WORKSPACE}

# ── 2. PyTorch ────────────────────────────────────────────────────
print("⚡ Installing PyTorch 2.8.0...")
!uv pip install --system torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --no-deps

# ── 3. Speed stack ────────────────────────────────────────────────
print("🚀 Installing speed stack (xformers, triton, sageattention)...")
!uv pip install --system --upgrade xformers triton sageattention

# ── 4. ComfyUI requirements ───────────────────────────────────────
print("📦 Installing ComfyUI requirements...")
!uv pip install --system -r requirements.txt

# ── 5. Core dependencies ──────────────────────────────────────────
print("📚 Installing core dependencies...")
!uv pip install --system \
    accelerate einops diffusers \
    "safetensors>=0.4.2" \
    aiohttp pyyaml Pillow scipy tqdm psutil \
    "tokenizers>=0.13.3" sentencepiece soundfile \
    "kornia>=0.7.1" spandrel torchsde \
    av comfy_aimdo comfy-kitchen \
    comfyui-workflow-templates comfyui-embedded-docs

# ── 6. Transformers / HuggingFace ────────────────────────────────
print("🤗 Installing transformers & huggingface-hub...")
!uv pip install --system \
    "transformers>=4.45.0,<4.57.0" \
    "huggingface-hub>=0.23.0,<1.0" \
    hf_transfer

# ── 7. CUDA memory optimization ──────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅ CUDA memory config set\n")

# ── 8. ComfyUI Manager ────────────────────────────────────────────
manager_path = f"{WORKSPACE}/custom_nodes/ComfyUI-Manager"
if not os.path.exists(manager_path):
    print("📥 Installing ComfyUI Manager...")
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager {manager_path}
else:
    print("🔄 Updating ComfyUI Manager...")
    !cd {manager_path} && git pull -q
print("✅ ComfyUI Manager ready\n")

# ── 9. Verify ─────────────────────────────────────────────────────
import importlib.metadata as meta
print("📋 Key package versions:")
for pkg in ["torch", "xformers", "transformers", "huggingface-hub", "safetensors"]:
    try:
        print(f"  ✅ {pkg}: {meta.version(pkg)}")
    except Exception:
        print(f"  ❌ {pkg}: not found")

print("\n🎉 Installation complete! Run the next cell.")


📦 Installing uv...
downloading uv 0.10.8 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
✅ uv ready

📥 Cloning ComfyUI...
✅ Cloned

/content/ComfyUI
⚡ Installing PyTorch 2.8.0...
Using Python 3.12.12 environment at: /usr
Resolved 3 packages in 47ms
Prepared 3 packages in 9.15s
Uninstalled 3 packages in 616ms
Installed 3 packages in 226ms
 - torch==2.10.0+cu128
 + torch==2.8.0
 - torchaudio==2.10.0+cu128
 + torchaudio==2.8.0
 - torchvision==0.25.0+cu128
 + torchvision==0.23.0
🚀 Installing speed stack (xformers, triton, sageattention)...
Using Python 3.12.12 environment at: /usr
Resolved 31 packages in 747ms
Prepared 7 packages in 9.73s
Uninstalled 5 packages in 336ms
Installed 7 packages in 244ms
 - filelock==3.24.3
 + filelock==3.25.0
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 - numpy==2.0.2
 + numpy==2.4.2
 + sageattention==1.0.6
 - setuptools==75.2.0
 + setuptools==82.0.0
 - torch==2.8.0
 + torch==2.10.0
 + xformers==0.0.3

In [4]:
# @title 🔧 3 · Custom Nodes { display-mode: "form" }
# @markdown Add GitHub repos to `CUSTOM_NODES` below and uncomment to install.
# @markdown Re-running is safe — existing nodes are skipped automatically.

import os, shutil

WORKSPACE        = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{WORKSPACE}/custom_nodes"

# ─── Add any GitHub URL here. Format: ("Folder_Name", "URL") ───
CUSTOM_NODES = [
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF"),
    # Add more nodes here...
]

# ─── INSTALLATION LOGIC ─────────────────────────────────────────
if not CUSTOM_NODES:
    print("ℹ️  No custom nodes configured — skipping.")
else:
    print("🚀 Starting Custom Node Installation...\n")
    for name, url in CUSTOM_NODES:
        path = os.path.join(CUSTOM_NODES_DIR, name)

        # Fix broken downloads from failed previous runs
        if os.path.exists(path) and not os.path.exists(os.path.join(path, ".git")):
            print(f"⚠️  Found broken folder '{name}', cleaning up...")
            shutil.rmtree(path)

        if not os.path.exists(path):
            print(f"📥 Cloning: {name}")
            !git clone {url} {path}
        else:
            print(f"⏭️  Already exists: {name}")

        req_file = os.path.join(path, "requirements.txt")
        if os.path.exists(req_file):
            print(f"📦 Installing deps for {name}...")
            !uv pip install --system -r {req_file}
        else:
            print(f"ℹ️  No requirements.txt for {name}")

    print("\n✨ All custom nodes processed.")

# ── Disable conflicting built-in node ────────────────────────────
websocket_node = f"{WORKSPACE}/custom_nodes/websocket_image_save.py"
if os.path.exists(websocket_node):
    os.rename(websocket_node, websocket_node + ".disabled")
    print("✅ Disabled websocket_image_save.py (was causing conflicts)")


🚀 Starting Custom Node Installation...

📥 Cloning: ComfyUI-GGUF
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-GGUF'...
remote: Enumerating objects: 814, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 814 (delta 457), reused 314 (delta 314), pack-reused 306 (from 2)
Receiving objects: 100% (814/814), 189.94 KiB | 1.53 MiB/s, done.
Resolving deltas: 100% (542/542), done.
📦 Installing deps for ComfyUI-GGUF...
Using Python 3.12.12 environment at: /usr
Resolved 11 packages in 77ms
Prepared 1 package in 25ms
Installed 1 package in 4ms
 + gguf==0.18.0

✨ All custom nodes processed.
✅ Disabled websocket_image_save.py (was causing conflicts)


In [2]:
# @title 📥 4 · Download Models { display-mode: "form" }
# @markdown Add as many models as you want in the list below.<br>
# @markdown - For **Hugging Face** → use `"type": "hf"` + `repo_id` + `filename`.<br>
# @markdown - For **Civitai / mirrors / other** → use `"type": "url"` + direct `url` + `filename`.

import os
from huggingface_hub import hf_hub_download, snapshot_download
from tqdm import tqdm
import subprocess

# Install aria2c for fast URL downloads
!apt-get update -qq && apt-get install -y -qq aria2
print("✅ aria2c installed\n")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import hf_transfer  # must import to activate fast downloads

# ────────────────────────────────────────────────
#   ↓↓↓  MODELS — edit here  ↓↓↓
# ────────────────────────────────────────────────

downloads = [
     # Qwen text encoder (GGUF version for low VRAM)
    {
        "type": "hf",
        "repo_id": "unsloth/Qwen2.5-VL-7B-Instruct-GGUF",
        "filename": "Qwen2.5-VL-7B-Instruct-Q5_K_M.gguf",  # ~10-12GB, good L4 balance
        "folder": "models/text_encoders",
    },
    # Multi-angle LoRA
    {
        "type": "hf",
        "repo_id": "Comfy-Org/Qwen-Image-Edit_ComfyUI",
        "filename": "split_files/loras/Qwen-Edit-2509-Multiple-angles.safetensors",
        "folder": "models/loras",
    },
    # Lightning 4-step model
    {
        "type": "hf",
        "repo_id": "lightx2v/Qwen-Image-Lightning",
        "filename": "Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors",
        "folder": "models/loras",
    },
    # Diffusion model
    {
        "type": "hf",
        "repo_id": "QuantStack/Qwen-Image-Edit-2509-GGUF",
        "filename": "Qwen-Image-Edit-2509-Q5_K_M.gguf",
        "folder": "models/unet",
    },
    # VAE
    {
        "type": "hf",
        "repo_id": "Comfy-Org/Qwen-Image_ComfyUI",
        "filename": "split_files/vae/qwen_image_vae.safetensors",
        "folder": "models/vae",
    },
     {
    "type": "hf",
    "repo_id": "unsloth/Qwen2.5-VL-7B-Instruct-GGUF",
    "filename": "mmproj-F16.gguf",
    "folder": "models/text_encoders",
},
]

# ────────────────────────────────────────────────
#   Download logic — no need to edit below
# ────────────────────────────────────────────────

base_dir = "/content/ComfyUI"

for item in tqdm(downloads, desc="Downloading models"):
    target_folder = os.path.join(base_dir, item["folder"].lstrip("/"))
    os.makedirs(target_folder, exist_ok=True)

    if item["type"].lower() in ["hf", "huggingface"]:
        repo  = item["repo_id"]
        file  = item.get("filename")
        fname = os.path.basename(file) if file else None
        dest  = os.path.join(target_folder, fname) if fname else target_folder
        print(f"\n📥 HF → {repo}  /  {file or 'FULL REPO'}")

        if os.path.exists(dest):
            print(f"   ⏭️  Already exists, skipping")
            continue

        try:
            if file is None:
                snapshot_download(repo_id=repo, local_dir=target_folder,
                                  local_dir_use_symlinks=False, resume_download=True)
            else:
                dl = hf_hub_download(repo_id=repo, filename=file,
                                     local_dir=target_folder, resume_download=True)
                # Flatten nested dirs if hf_hub created subdirectories
                if os.path.dirname(dl) != target_folder and os.path.exists(dl):
                    import shutil
                    shutil.move(dl, dest)
                    try:
                        os.removedirs(os.path.dirname(dl))
                    except Exception:
                        pass
            print(f"   ✅ Done")
        except Exception as e:
            print(f"   ⚠️  HF download error: {e}")

    elif item["type"].lower() == "url":
        url   = item["url"]
        fname = item["filename"]
        dest  = os.path.join(target_folder, fname)
        print(f"\n📥 URL → {fname}")

        if os.path.exists(dest):
            print(f"   ⏭️  Already exists, skipping")
            continue

        cmd = ["aria2c", "--console-log-level=error", "-c",
               "-x", "16", "-s", "16", "-j", "8", "-k", "1M",
               url, "-d", target_folder, "-o", fname]
        try:
            subprocess.run(cmd, check=True)
            print(f"   ✅ Done")
        except Exception as e:
            print(f"   ⚠️  aria2c failed: {e}")
    else:
        print(f"⚠️  Unknown type '{item['type']}' — skipping")

print("\n✅ All downloads finished!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ aria2c installed




📥 HF → unsloth/Qwen2.5-VL-7B-Instruct-GGUF  /  Qwen2.5-VL-7B-Instruct-Q5_K_M.gguf
   ⏭️  Already exists, skipping

📥 HF → Comfy-Org/Qwen-Image-Edit_ComfyUI  /  split_files/loras/Qwen-Edit-2509-Multiple-angles.safetensors
   ⏭️  Already exists, skipping

📥 HF → lightx2v/Qwen-Image-Lightning  /  Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors
   ⏭️  Already exists, skipping

📥 HF → QuantStack/Qwen-Image-Edit-2509-GGUF  /  Qwen-Image-Edit-2509-Q5_K_M.gguf
   ⏭️  Already exists, skipping

📥 HF → Comfy-Org/Qwen-Image_ComfyUI  /  split_files/vae/qwen_image_vae.safetensors
   ⏭️  Already exists, skipping

📥 HF → unsloth/Qwen2.5-VL-7B-Instruct-GGUF  /  mmproj-F16.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


mmproj-F16.gguf:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

   ✅ Done

✅ All downloads finished!


In [3]:
!mv /content/ComfyUI/models/text_encoders/mmproj-F16.gguf \
   /content/ComfyUI/models/text_encoders/Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf

In [ ]:
# @title 🚀 5 · Launch ComfyUI { display-mode: "form" }
# @markdown ## Launch ComfyUI + Multi-Angle Batch Watcher
# @markdown
# @markdown **Launch modes:**
# @markdown - `window` — opens ComfyUI in a new browser tab (default)
# @markdown - `iframe` — embeds ComfyUI inside the notebook cell
# @markdown - `cloudflare` — public URL via Cloudflare tunnel
# @markdown
# @markdown Drop images into your Drive `input/` folder to trigger processing automatically.
# @markdown Set `BATCH_MODE = False` to just launch ComfyUI without the watcher.

import os, time, json, shutil, threading, socket, glob, requests, uuid, subprocess
from pathlib import Path
from datetime import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import builtins
if getattr(builtins, "_comfyui_launched", False):
    print("⚠️  Already running. Use Runtime → Restart to relaunch.")
    raise SystemExit()
builtins._comfyui_launched = True

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                              ║
# ╚══════════════════════════════════════════════════════════════════╝

LAUNCH_MODE = "window"   # @param ["window", "iframe", "cloudflare"]
BATCH_MODE  = True       # @param {type:"boolean"}

# Full absolute path — examples:
# My Drive    : /content/drive/MyDrive/comfyui/multi-angle
# Shared Drive: /content/drive/Shareddrives/Figuro/multi-angle-shots
DRIVE_BASE_PATH   = "/content/drive/Shareddrives/Figuro/multi-angle-shots"  # @param {type:"string"}
EXTRA_OUTPUT_PATH = "/content/drive/Shareddrives/Figuro/image-upscale"      # @param {type:"string"}

WORKFLOW_FILENAME    = "workflow_api.json"   # @param {type:"string"}
INPUT_NODE_ID        = "25"                  # @param {type:"string"}
POLL_INTERVAL        = 5                     # @param {type:"integer"}
SUPPORTED_EXTENSIONS = "png,jpg,jpeg"        # @param {type:"string"}

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧  PATH SETUP                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive as _drive
if not os.path.exists("/content/drive/MyDrive") and not os.path.exists("/content/drive/Shareddrives"):
    print("📂 Drive not mounted — mounting now...")
    _drive.mount('/content/drive', force_remount=False)

def validate_path(label, path):
    path = path.strip().rstrip("/")
    if not path.startswith("/content/drive/"):
        raise ValueError(
            f"❌ {label} must start with /content/drive/MyDrive/... "
            f"or /content/drive/Shareddrives/<n>/...\n   Got: '{path}'"
        )
    os.makedirs(path, exist_ok=True)
    return path

def safe_makedirs(path):
    try:
        os.makedirs(path, exist_ok=True)
        return path
    except OSError as e:
        fallback = path.replace("/content/drive", "/tmp/drive_mirror")
        os.makedirs(fallback, exist_ok=True)
        print(f"⚠️  Cannot create '{path}': {e}")
        print(f"   → Using local fallback: '{fallback}'")
        return fallback

COMFYUI_URL  = "http://127.0.0.1:8188"
COMFY_OUTPUT = "/content/ComfyUI/output"
EXTENSIONS   = {e.strip().lower().lstrip('.') for e in SUPPORTED_EXTENSIONS.split(',') if e.strip()}
_FILE_FIELDS = ["image", "video", "audio", "file", "mask",
                "image_path", "video_path", "file_path", "input", "source", "path"]

if BATCH_MODE:
    MAIN_FOLDER   = validate_path("DRIVE_BASE_PATH", DRIVE_BASE_PATH)
    INPUT_FOLDER  = safe_makedirs(os.path.join(MAIN_FOLDER, "input"))
    PROCESSED_DIR = safe_makedirs(os.path.join(MAIN_FOLDER, "processed"))
    OUTPUT_DIR    = safe_makedirs(os.path.join(MAIN_FOLDER, "output"))
    WORKFLOW_PATH = os.path.join(MAIN_FOLDER, WORKFLOW_FILENAME)
    EXTRA_DIR     = None
    if EXTRA_OUTPUT_PATH.strip():
        EXTRA_DIR = validate_path("EXTRA_OUTPUT_PATH", EXTRA_OUTPUT_PATH.strip())
    print(f"✅ Paths ready:")
    print(f"   Input     : {INPUT_FOLDER}")
    print(f"   Processed : {PROCESSED_DIR}")
    print(f"   Output    : {OUTPUT_DIR}")
    print(f"   Workflow  : {WORKFLOW_PATH}")
    if EXTRA_DIR:
        print(f"   Extra out : {EXTRA_DIR}")
else:
    MAIN_FOLDER = WORKFLOW_PATH = INPUT_FOLDER = PROCESSED_DIR = OUTPUT_DIR = EXTRA_DIR = None

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖  BATCH WATCHER HELPERS                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

def load_workflow():
    if not os.path.exists(WORKFLOW_PATH):
        raise FileNotFoundError(f"Workflow not found: {WORKFLOW_PATH}")
    with open(WORKFLOW_PATH) as f:
        return json.load(f)

def auto_detect_field(workflow, node_id):
    if node_id not in workflow:
        raise KeyError(f"Node '{node_id}' not in workflow. Available: {list(workflow.keys())}")
    inputs = workflow[node_id].get("inputs", {})
    for known in _FILE_FIELDS:
        if known in inputs:
            return known
    media_exts = {"png","jpg","jpeg","webp","gif","bmp","tiff","tif","mp4","avi","mov"}
    for field, val in inputs.items():
        if isinstance(val, str) and Path(val).suffix.lower().lstrip('.') in media_exts:
            return field
    short = [f for f, v in inputs.items()
             if isinstance(v, str) and len(v) < 256 and not v.startswith(("http", "{"))]
    if short:
        return short[0]
    raise ValueError(f"Can't detect input field on node '{node_id}'. Inputs: {list(inputs.keys())}")

def wait_for_comfyui(timeout=120):
    print("⏳ Waiting for ComfyUI...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(f"{COMFYUI_URL}/system_stats", timeout=3).status_code == 200:
                print(" ✅ Ready!")
                return
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(2)
    raise TimeoutError("ComfyUI did not start in time.")

def queue_prompt(workflow):
    client_id = str(uuid.uuid4())
    r = requests.post(f"{COMFYUI_URL}/prompt",
                      json={"prompt": workflow, "client_id": client_id}, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"Prompt error: {data['error']}")
    return data["prompt_id"]

def poll_until_done(prompt_id, timeout=600):
    deadline = time.time() + timeout
    while time.time() < deadline:
        time.sleep(3)
        try:
            history = requests.get(f"{COMFYUI_URL}/history/{prompt_id}", timeout=10).json()
        except Exception:
            continue
        if prompt_id not in history:
            continue
        if history[prompt_id].get("outputs") is not None:
            return True
    raise TimeoutError(f"Prompt {prompt_id} timed out after {timeout}s")

def clear_comfy_output():
    """Wipe /content/ComfyUI/output before each job so only new files are synced."""
    if os.path.exists(COMFY_OUTPUT):
        for f in os.listdir(COMFY_OUTPUT):
            fp = os.path.join(COMFY_OUTPUT, f)
            if os.path.isfile(fp):
                os.remove(fp)

def sync_comfy_output_to_drive():
    """Copy all files from /content/ComfyUI/output → Drive output + extra dirs."""
    dest_dirs = [OUTPUT_DIR] + ([EXTRA_DIR] if EXTRA_DIR else [])
    synced = 0
    for fname in sorted(os.listdir(COMFY_OUTPUT)):
        src = os.path.join(COMFY_OUTPUT, fname)
        if not os.path.isfile(src):
            continue
        for dest in dest_dirs:
            shutil.copy2(src, os.path.join(dest, fname))
        print(f"   💾 {fname}")
        synced += 1
    if synced == 0:
        print("   ⚠️  No outputs found — check workflow has a Save Image node.")
    return synced

def copy_input_to_comfyui(src_path):
    comfyui_input = "/content/ComfyUI/input"
    os.makedirs(comfyui_input, exist_ok=True)
    filename = Path(src_path).name
    shutil.copy2(src_path, os.path.join(comfyui_input, filename))
    return filename

def process_file(filepath):
    stem = Path(filepath).stem
    print(f"\n{'─'*55}")
    print(f"📂 {Path(filepath).name}  [{datetime.now().strftime('%H:%M:%S')}]")

    clear_comfy_output()  # wipe output so only THIS job's files get synced

    workflow = load_workflow()
    field    = auto_detect_field(workflow, INPUT_NODE_ID)
    filename = copy_input_to_comfyui(filepath)
    workflow[INPUT_NODE_ID]["inputs"][field] = filename
    print(f"   Node [{INPUT_NODE_ID}].{field} → '{filename}'")

    prompt_id = queue_prompt(workflow)
    print(f"   Queued: {prompt_id}")
    print("   ⏳ Running...", end="", flush=True)
    poll_until_done(prompt_id)
    print(f" done")

    synced = sync_comfy_output_to_drive()
    print(f"   ✅ {synced} file(s) saved to Drive")

    dest_proc = os.path.join(PROCESSED_DIR, Path(filepath).name)
    if os.path.exists(dest_proc):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest_proc = os.path.join(PROCESSED_DIR, f"{stem}_{ts}{Path(filepath).suffix}")
    shutil.move(filepath, dest_proc)
    print(f"   📦 Moved to processed")

def run_watcher():
    wait_for_comfyui()
    try:
        wf    = load_workflow()
        field = auto_detect_field(wf, INPUT_NODE_ID)
        cls   = wf[INPUT_NODE_ID].get("class_type", "unknown")
        print(f"\n✅ Workflow: {WORKFLOW_PATH}")
        print(f"   Node [{INPUT_NODE_ID}] {cls} → field: '{field}'")
    except Exception as e:
        print(f"\n⚠️  Workflow check failed: {e}")
        return
    print(f"\n👀 Watching '{INPUT_FOLDER}' every {POLL_INTERVAL}s")
    print("   Drop images into the input folder to trigger the workflow.\n")
    seen_errors = {}
    while True:
        try:
            candidates = sorted(set(
                f for ext in EXTENSIONS
                for f in glob.glob(os.path.join(INPUT_FOLDER, f"*.{ext}")) +
                          glob.glob(os.path.join(INPUT_FOLDER, f"*.{ext.upper()}"))
            ))
            for fp in candidates:
                if seen_errors.get(fp, 0) >= 3:
                    continue
                try:
                    process_file(fp)
                    seen_errors.pop(fp, None)
                except Exception as e:
                    seen_errors[fp] = seen_errors.get(fp, 0) + 1
                    print(f"\n❌ Error ({seen_errors[fp]}/3) — {Path(fp).name}: {e}")
                    if seen_errors[fp] >= 3:
                        print(f"   ⛔ Giving up on {Path(fp).name}")
        except Exception as e:
            print(f"⚠️  Watcher error: {e}")
        time.sleep(POLL_INTERVAL)

# ── Launch modes ──────────────────────────────────────────────────

def start_window(port):
    while True:
        time.sleep(0.5)
        s = socket.socket()
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close(); break
        s.close()
    from google.colab import output as co
    print("\n🌐 ComfyUI is ready!")
    co.serve_kernel_port_as_window(port)

def start_iframe(port):
    while True:
        time.sleep(0.5)
        s = socket.socket()
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close(); break
        s.close()
    from google.colab import output as co
    print("\n🌐 ComfyUI is ready!")
    co.serve_kernel_port_as_iframe(port, height=900)
    co.serve_kernel_port_as_window(port)

def start_cloudflare(port):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], capture_output=True)
    while True:
        time.sleep(0.5)
        s = socket.socket()
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close(); break
        s.close()
    print("\n🌐 Launching Cloudflare tunnel...")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com" in l:
            print("ComfyUI URL:", l[l.find("http"):], end='')

# ── Summary & start ───────────────────────────────────────────────
print("=" * 55)
print("🎨  Qwen Multi-Angle Batch Runner")
print("=" * 55)
print(f"  Mode       : {LAUNCH_MODE} | Batch: {BATCH_MODE}")
if BATCH_MODE:
    print(f"  Input      : {INPUT_FOLDER}")
    print(f"  Output     : {OUTPUT_DIR}")
    print(f"  Workflow   : {WORKFLOW_PATH}")
    print(f"  Node ID    : {INPUT_NODE_ID}")
    print(f"  Poll       : {POLL_INTERVAL}s")
    if EXTRA_DIR:
        print(f"  Extra out  : {EXTRA_DIR}")
print("=" * 55)

%cd /content/ComfyUI

if LAUNCH_MODE == "iframe":
    threading.Thread(target=start_iframe,     daemon=True, args=(8188,)).start()
elif LAUNCH_MODE == "cloudflare":
    threading.Thread(target=start_cloudflare, daemon=True, args=(8188,)).start()
else:
    threading.Thread(target=start_window,     daemon=True, args=(8188,)).start()

if BATCH_MODE:
    threading.Thread(target=run_watcher, daemon=True).start()

print("\n🚀 Starting ComfyUI...\n")
!python main.py --highvram --cuda-malloc --dont-print-server

✅ Paths ready:
   Input     : /content/drive/Shareddrives/Figuro/multi-angle-shots/input
   Processed : /content/drive/Shareddrives/Figuro/multi-angle-shots/processed
   Output    : /content/drive/Shareddrives/Figuro/multi-angle-shots/output
   Workflow  : /content/drive/Shareddrives/Figuro/multi-angle-shots/workflow_api.json
   Extra out : /content/drive/Shareddrives/Figuro/image-upscale
🎨  Qwen Multi-Angle Batch Runner
  Mode       : window | Batch: True
  Input      : /content/drive/Shareddrives/Figuro/multi-angle-shots/input
  Output     : /content/drive/Shareddrives/Figuro/multi-angle-shots/output
  Workflow   : /content/drive/Shareddrives/Figuro/multi-angle-shots/workflow_api.json
  Node ID    : 25
  Poll       : 5s
  Extra out  : /content/drive/Shareddrives/Figuro/image-upscale
/content/ComfyUI
⏳ Waiting for ComfyUI...
🚀 Starting ComfyUI...

.[START] Security scan
.[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-03-06 13:01:38

<IPython.core.display.Javascript object>

 ✅ Ready!

✅ Workflow: /content/drive/Shareddrives/Figuro/multi-angle-shots/workflow_api.json
   Node [25] LoadImage → field: 'image'

👀 Watching '/content/drive/Shareddrives/Figuro/multi-angle-shots/input' every 5s
   Drop images into the input folder to trigger the workflow.

FETCH ComfyRegistry Data: 5/128
FETCH ComfyRegistry Data: 10/128
FETCH ComfyRegistry Data: 15/128
FETCH ComfyRegistry Data: 20/128
FETCH ComfyRegistry Data: 25/128

───────────────────────────────────────────────────────
📂 Gemini_Generated_Image_qfrqbzqfrqbzqfrq.png  [13:02:04]
   Node [25].image → 'Gemini_Generated_Image_qfrqbzqfrqbzqfrq.png'
   Queued: 227c578d-ff85-4aae-98cd-19d8f1f8a6dc
   ⏳ Running...got prompt
Using xformers attention in VAE
Using xformers attention in VAE
VAE load device: cuda:0, offload device: cpu, dtype: torch.bfloat16
Requested to load WanVAE
loaded completely;  242.03 MB loaded, full load: True
FETCH ComfyRegistry Data: 30/128
FETCH ComfyRegistry Data: 35/128
FETCH ComfyRegistry Data

---

## 📝 Notes

**Workflow format** — Export via ComfyUI → Settings → Dev Mode → *Save (API Format)*. Uses numeric node IDs as top-level keys.

**Input node ID** — open `workflow_api.json`, find your Load Image node, its top-level key (e.g. `"25"`) is `INPUT_NODE_ID`. The field is auto-detected and printed on startup.

**Batch mode off** — set `BATCH_MODE = False` to just launch ComfyUI without the watcher.

**Launch modes** — `window` opens a new tab (recommended), `iframe` embeds in the cell, `cloudflare` gives a public shareable URL.

**Error handling** — files that fail 3 times are skipped permanently so the watcher never loops.

**Live edits** — workflow JSON is re-read on every file so you can tweak it without restarting.

**Models included:**
- `qwen_2.5_vl_7b_fp8_scaled` — text encoder
- `Qwen-Edit-2509-Multiple-angles` — multi-angle LoRA
- `Qwen-Image-Edit-2509-Lightning-4steps` — 4-step fast inference
- `qwen_image_edit_2509_fp8_e4m3fn` — diffusion model
- `qwen_image_vae` — VAE

---
